# Lab: Visualizing Limits with Graphs + Tables (APEX 1.1) — **SymPy-First Edition**

In this lab you’ll build intuition for limits using three tools *together*:

- **Sketches**: you draw a quick picture first (don’t skip this!)
- **Tables**: you approximate the limit numerically from the **left** and **right**
- **Plots**: you visualize what the function is doing near the point of interest

We’ll start with the classic function **$f(x)=\frac{\sin(x)}{x}$**, then work through **Examples 1.1.1–1.1.5**.

---

## Learning Targets (VARIOSA)
- **V**: visualize limit behavior with plots
- **A**: approximate limits with left/right tables
- **S**: represent functions symbolically and (optionally) compute exact limits
- **R**: explain *why* a limit exists or does not exist

> **Workflow for each example:** **Sketch → Plot → Tables → Explain.**


We'll be exploring functions from the examples in the textbook with plots and tables. First let's create functions that make the tables easy to create and learn from. **Be sure to run this first!**

In [ ]:
import numpy as np
import sympy as sp
from sympy.plotting import plot

# Pretty printing for SymPy in Jupyter
sp.init_printing()

# We'll use a single symbol x throughout the notebook
x = sp.Symbol('x')

def make_table(x_vals, f_num):
    """Create a list of (x, f(x)) pairs for a numeric function f_num."""
    rows = []
    for xv in x_vals:
        try:
            yv = float(f_num(xv))  # force float for consistent printing
            rows.append((xv, yv))
        except Exception:
            rows.append((xv, None))
    return rows

def print_table(rows, x_label="x", y_label="f(x)", digits=6):
    """Pretty-print a small table of x and y values."""
    print(f"{x_label:>12}  {y_label:>16}")
    print("-"*30)
    for xv, yv in rows:
        if yv is None or (isinstance(yv, float) and (np.isnan(yv) or np.isinf(yv))):
            y_str = "undefined"
        else:
            y_str = f"{yv:.{digits}f}"
        print(f"{xv:>12}  {y_str:>16}")

def plot_split_interval(expr, left_range, right_range, title="", xlabel="x", ylabel="y",
                        left_color="blue", right_color="red", ylim=None):
    """Plot a SymPy expression on two x-intervals (to avoid a problematic point).

    This avoids the SymPy error about 'Too many ranges' by:
    1) creating a plot on the left interval
    2) creating a plot on the right interval
    3) appending the right series to the left plot container

    Colors help students visually separate behavior from the left vs right.
    """
    p = plot(expr, (x, left_range[0], left_range[1]), show=False,
             title=title, xlabel=xlabel, ylabel=ylabel,
             line_color=left_color)
    p_right = plot(expr, (x, right_range[0], right_range[1]), show=False,
                   line_color=right_color)
    p.append(p_right[0])
    if ylim is not None:
        p.ylim = ylim
    p.show()


## 1) Warm-up: $$f(x)=\sin(x)/x$$ near $x=1$

### ✏️ Sketch (do this before running code)
1. Evaluate the function at 1. What is f(1)?
2. Sketch $y=\sin(x)/x$ near $x=1$.
3. Predict the value of the limit:  
$$\lim_{x\to 1}\frac{\sin(x)}{x}$$

Now we’ll plot the function and build two tables: one from the left and one from the right.

**Play around with the viewing window of the graph, as well as the values in the table to help you understand the limit.**


In [ ]:
# Step 1: Define the function symbolically
f = sp.sin(x)/x

# Step 2: Convert to a numeric function for table evaluation
f_num = sp.lambdify(x, f, 'numpy')

# Step 3: Plot near x=1 (no split needed here)
p = plot(f, (x, -1, 3), show=False,
         title=r"Plot of $\sin(x)/x$ near $x=1$",
         xlabel="x", ylabel="y",
         line_color="blue")
p.show()

# Step 4: Tables approaching 1 from both sides
print("Limit from the left (x → 1⁻)")
x_vals1 = [0.9, 0.99, 0.999]
rows = make_table(x_vals1, f_num)
print_table(rows, y_label="sin(x)/x", digits=10)
print("################################")

print("Limit from the right (x → 1⁺)")
x_vals2 = [1.1, 1.01, 1.001]
rows = make_table(x_vals2, f_num)
print_table(rows, y_label="sin(x)/x", digits=10)

print("\nYour estimate of the limit (write it here): ____")

## 2) The classic: $\sin(x)/x$ near $x=0$

At $x=0$, $\sin(x)/x$ is **undefined** (it becomes $0/0$).  
But a limit can still exist if the values approach a single number as $x$ gets close to $0$.

### ✏️ Sketch
1. Evaluate the function at 0. What is f(0)?
2. Sketch $y=\sin(x)/x$ near $x=0$.
3. Predict:  
$$\lim_{x\to 0}\frac{\sin(x)}{x}$$

We’ll split the plot into two intervals so we never evaluate at $x=0$.

**Play around with the viewing window of the graph (`left_range` and `right_range`), as well as the values in the table to help you understand the limit.**


In [ ]:
f = sp.sin(x)/x
f_num = sp.lambdify(x, f, 'numpy')

# Split-interval plot: left side (blue) and right side (red)
plot_split_interval(
    expr=f,
    left_range=(-10, -0.001),
    right_range=(0.001, 10),
    title=r"Plot of $\sin(x)/x$ near $x=0$ (left = blue, right = red)",
    xlabel="x",
    ylabel="y"
)

# Tables
print("Limit from the left (x → 0⁻)")
x_vals1 = [-0.1, -0.01, -0.001]
rows = make_table(x_vals1, f_num)
print_table(rows, y_label="sin(x)/x", digits=12)
print("################################")

print("Limit from the right (x → 0⁺)")
x_vals2 = [0.1, 0.01, 0.001]
rows = make_table(x_vals2, f_num)
print_table(rows, y_label="sin(x)/x", digits=12)

print("\nYour estimate of the limit (write it here): ____")

### Exact check with SymPy

After you’ve predicted the limit, use SymPy to compute it exactly:


In [ ]:
sp.limit(sp.sin(x)/x, x, 0)

## 3) Example 1.1.6: A rational function limit near a “hole”

Approximate:
$$\lim_{x\to 3}\frac{x^2-x-6}{6x^2-19x+3}$$

### ✏️ Sketch
1. Evaluate the function at 3. What is f(3)?
2. Sketch the function near $x=3$.
3. Does it look like it approaches a single $y$-value?
4. Do you see evidence of a “hole” (a removable discontinuity)?
**5. IMPORTANT!!! - Do the math to "PROVE" your answer**


**Play around with the viewing window of the graph (`left_range` and `right_range`), as well as the values in the table to help you understand the limit.**

In [ ]:
# Symbolic definition
f = (x**2 - x - 6)/(6*x**2 - 19*x + 3)
f_num = sp.lambdify(x, f, 'numpy')

# Plot split around x=3
plot_split_interval(
    expr=f,
    left_range=(-10, 2.999),
    right_range=(3.001, 10),
    title=r"Example 1.1.1 near $x=3$ (left = blue, right = red)",
    xlabel="x",
    ylabel="value"
)

# Two separate tables
print("Limit from the left (x → 3⁻)")
x_vals1 = [2.9, 2.99, 2.999]
rows = make_table(x_vals1, f_num)
print_table(rows, y_label="value", digits=10)
print("################################")

print("Limit from the right (x → 3⁺)")
x_vals2 = [3.1, 3.01, 3.001]
rows = make_table(x_vals2, f_num)
print_table(rows, y_label="value", digits=10)

print("\nYour estimate of the limit (write it here): ____")

### Exact limit with SymPy

Use this after you’ve made your numerical/graphical prediction.


In [ ]:
sp.factor(f), sp.limit(f, x, 3)

## 4) Example 1.1.9: Piecewise function near $x=0$

Define
$$
f(x)=\begin{cases}
x+1 & x<0\\
-x^2+1 & x>0
\end{cases}
$$
(Notice $f(0)$ is not defined.)

### ✏️ Sketch
1. Evaluate the function at 0. What is f(0)?
2. Sketch both pieces.
3. What value does the left-hand side approach as $x\to 0^-$?
4. What value does the right-hand side approach as $x\to 0^+$?

**Play around with the viewing window of the graph (`left_range` and `right_range`), as well as the values in the table to help you understand the limit.**

In [ ]:
f = sp.Piecewise((x + 1, x < 0), (-x**2 + 1, x > 0))
f_num = sp.lambdify(x, f, 'numpy')

plot_split_interval(
    expr=f,
    left_range=(-10, -0.001),
    right_range=(0.001, 10),
    title="Example 1.1.2 near x=0 (left = blue, right = red)",
    xlabel="x",
    ylabel="f(x)"
)

print("Limit from the left (x → 0⁻)")
x_vals1 = [-0.1, -0.01, -0.001]
rows = make_table(x_vals1, f_num)
print_table(rows, y_label="f(x)", digits=10)
print("################################")

print("Limit from the right (x → 0⁺)")
x_vals2 = [0.1, 0.01, 0.001]
rows = make_table(x_vals2, f_num)
print_table(rows, y_label="f(x)", digits=10)

print("\nYour estimate of the limit (write it here): ____")

## 5) Example 1.1.12: Left and right limits disagree

$$
f(x)=\begin{cases}
x^2-2x+3 & x\le 1\\
x & x>1
\end{cases}
$$

### ✏️ Sketch
1. Evaluate the function at 1. What is f(1)?
2. Sketch both pieces near $x=1$.
3. Compare the left-hand limit and right-hand limit.
4. Decide: does the two-sided limit exist?

**Play around with the viewing window of the graph (`left_range` and `right_range`), as well as the values in the table to help you understand the limit.**


In [ ]:
f = sp.Piecewise((x**2 - 2*x + 3, x <= 1), (x, x > 1))
f_num = sp.lambdify(x, f, 'numpy')

plot_split_interval(
    expr=f,
    left_range=(-10, 0.999),
    right_range=(1.001, 10),
    title="Example 1.1.3 near x=1 (left = blue, right = red)",
    xlabel="x",
    ylabel="f(x)"
)

print("Limit from the left (x → 1⁻)")
x_vals1 = [0.9, 0.99, 0.999]
rows = make_table(x_vals1, f_num)
print_table(rows, y_label="f(x)", digits=10)
print("################################")

print("Limit from the right (x → 1⁺)")
x_vals2 = [1.1, 1.01, 1.001]
rows = make_table(x_vals2, f_num)
print_table(rows, y_label="f(x)", digits=10)

print("\nDoes the two-sided limit exist? Explain in one sentence:")

## 6) Example 1.1.15: Grows without bound

Explore:
$$\lim_{x\to 1}\frac{1}{(x-1)^2}$$

This is a case where the function values grow without bound near $x=1$.

### ✏️ Sketch
1. Evaluate the function at 1. What is f(1)?
2. Sketch $1/(x-1)^2$ near $x=1$.
3. What do you expect happens from the left and from the right?

**Play around with the viewing window of the graph (`left_range` and `right_range`), as well as the values in the table to help you understand the limit.**


In [ ]:
f = 1/(x - 1)**2
f_num = sp.lambdify(x, f, 'numpy')

plot_split_interval(
    expr=f,
    left_range=(-10, 0.999),
    right_range=(1.001, 10),
    title=r"Example 1.1.4 near $x=1$ (left = blue, right = red)",
    xlabel="x",
    ylabel="value",
    ylim=(0, 120)  # keep the plot readable
)

print("Limit from the left (x → 1⁻)")
x_vals1 = [0.9, 0.99, 0.999]
rows = make_table(x_vals1, f_num)
print_table(rows, y_label="1/(x-1)^2", digits=3)
print("################################")

print("Limit from the right (x → 1⁺)")
x_vals2 = [1.1, 1.01, 1.001]
rows = make_table(x_vals2, f_num)
print_table(rows, y_label="1/(x-1)^2", digits=3)

print("\nExplain in words: why is there no finite limit?")

## 7) Example 1.1.18: Oscillation (no single value approached)

Explore:
$$\lim_{x\to 0}\sin(1/x)$$

This function keeps bouncing between $-1$ and $1$ as $x$ gets closer to $0$.
The oscillations also happen faster and faster.

### ✏️ Sketch
1. Evaluate the function at 0. What is f(0)?
2. Sketch $\sin(1/x)$ near $x=0$ (qualitatively).
3. What would it mean for a limit to exist here?

**Play around with the viewing window of the graph (`left_range` and `right_range`), as well as the values in the table to help you understand the limit.**

In [ ]:
f = sp.sin(1/x)
f_num = sp.lambdify(x, f, 'numpy')

# Wide view
plot_split_interval(
    expr=f,
    left_range=(-10, -0.001),
    right_range=(0.001, 10),
    title=r"Example 1.1.5: $\sin(1/x)$ on [-1,1] (left = blue, right = red)",
    xlabel="x",
    ylabel="value"
)

# Zoomed view
plot_split_interval(
    expr=f,
    left_range=(-10, -0.0005),
    right_range=(0.0005, 10),
    title=r"Zoom: $\sin(1/x)$ near 0 (left = blue, right = red)",
    xlabel="x",
    ylabel="value"
)

print("Limit from the left (x → 0⁻)")
x_vals1 = [-0.1, -0.01, -0.001, -0.0001]
rows = make_table(x_vals1, f_num)
print_table(rows, y_label="sin(1/x)", digits=6)
print("################################")

print("Limit from the right (x → 0⁺)")
x_vals2 = [0.1, 0.01, 0.001, 0.0001]
rows = make_table(x_vals2, f_num)
print_table(rows, y_label="sin(1/x)", digits=6)

print("\nExplain in words: why does this prevent a limit from existing?")

## Wrap-up Reflection

Answer in complete sentences:

1. For each example, did the limit exist? If yes, what was it approximately (and exactly, when you checked with SymPy)?
2. Why is approaching from both sides (left and right) important?
3. Give one data science interpretation of “approximation instead of exactness.”
   - Examples: noisy measurements, rounding, floating-point error, “nearby points” in kNN, etc.
